# NB_lab_test_trends

Builds `adb_db_caresync.gold.lab_test_trends` from the silver layer per the STTM (`Gold/sttm_silver_to_gold.xlsx`).

* **Grain:** one row per test type (`test_name`) per calendar month
* **Driving table:** `silver.lab_results` grouped by `test_name`, year and month of `test_date`
* **Joins (LEFT, optional):** `silver.calendar` on `date_key = yyyyMMdd(date_trunc('month', test_date))` for `month_name`, `quarter_name`
* **Window:** `LAG(abnormal_rate_pct)` by `test_name` ordered by year, month
* **Load:** full rebuild with `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
%sql
CREATE OR REPLACE TABLE adb_db_caresync.gold.lab_test_trends AS
WITH monthly AS (
  SELECT
    test_name,
    YEAR(test_date)                             AS year,
    MONTH(test_date)                            AS month,
    CAST(date_trunc('month', test_date) AS DATE) AS month_start,
    COUNT(*)                                    AS total_tests,
    COUNT_IF(result_status = 'Abnormal')        AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')        AS critical_test_count
  FROM adb_db_caresync.silver.lab_results
  GROUP BY test_name, YEAR(test_date), MONTH(test_date), CAST(date_trunc('month', test_date) AS DATE)
),
rates AS (
  SELECT
    *,
    CAST(abnormal_test_count / total_tests * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(critical_test_count / total_tests * 100 AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly
),
trend AS (
  SELECT
    *,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates
)
SELECT
  t.test_name,
  t.year,
  t.month,
  c.month_name,
  c.quarter_name,
  CAST(t.total_tests AS INT)         AS total_tests,
  CAST(t.abnormal_test_count AS INT) AS abnormal_test_count,
  CAST(t.critical_test_count AS INT) AS critical_test_count,
  t.abnormal_rate_pct,
  t.critical_rate_pct,
  t.prev_month_abnormal_rate_pct,
  CAST(t.abnormal_rate_pct - t.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM trend t
LEFT JOIN adb_db_caresync.silver.calendar c
  ON c.date_key = CAST(date_format(t.month_start, 'yyyyMMdd') AS INT)

In [0]:
updated_df = spark.table("adb_db_caresync.gold.lab_test_trends")
record_count = updated_df.count()

dbutils.notebook.exit(str(record_count))